In [ ]:
import pandas as pd
import scipy.stats as st
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import re
import os
from dotenv import load_dotenv
from google import genai
from elevenlabs.client import ElevenLabs

In [ ]:
vul = pd.read_csv("data/vulnerabilities.csv.gz")
exploitSignals = pd.read_csv("data/exploit_signals_2024.csv.gz")
exploitSignals2 = pd.read_csv("data/exploit_signals_2025.csv.gz")
knownExploits = pd.read_csv("data/known_exploited.csv.gz")
scoreBoard = pd.read_csv("data/scoreboard_2026.csv.gz")
weaknesses = pd.read_csv("data/weakness_types.csv.gz")
ref = pd.read_csv("data/references.csv.gz") 
affects = pd.read_csv("data/affected_products.csv.gz")

In [ ]:
print(torch.cuda.is_available())
affects = pd.read_csv(
    "data/affected_products.csv.gz",
    dtype={
        "version": str,
        "update": str,
        "version_start_including": str,
        "version_start_excluding": str,
        "version_end_including": str,
        "version_end_excluding": str,
    },
    low_memory=False,
)

In [ ]:
# Combine 2024 and 2025 monthly signals
signals = pd.concat([exploitSignals, exploitSignals2], ignore_index=True)

# Parse KEV date_added to define your training ground truth
knownExploits["date_added"] = pd.to_datetime(knownExploits["date_added"])
kev_dates = dict(zip(knownExploits["cve_id"], knownExploits["date_added"]))

# Convert signal month (YYYY-MM) to month-end timestamp
signals["month_end"] = pd.to_datetime(signals["month"]) + pd.offsets.MonthEnd(1)

# Map KEV addition date
signals["kev_date"] = signals["cve_id"].map(kev_dates)

# Filter out rows where the CVE was ALREADY on KEV on or before the snapshot date
signals = signals[signals["in_kev"] == 0].copy()

# Label = 1 if added to KEV within 90 days after this snapshot month
signals["days_until_kev"] = (signals["kev_date"] - signals["month_end"]).dt.days
signals["target"] = (
    (signals["days_until_kev"] > 0) & (signals["days_until_kev"] <= 90)
).astype(int)

print("Positive samples in training window:", signals["target"].sum())

In [ ]:
# Safe static columns available from the initial disclosure
safe_vul_cols = [
    "cve_id",
    "description",
    "assigner",
    "cwe_ids",
    "attack_vector",
    "attack_complexity",
    "privileges_required",
    "user_interaction",
    "cvss3_score",
]

# Keep only columns present in the dataset
available_cols = [c for c in safe_vul_cols if c in vul.columns]
vul_clean = vul[available_cols].copy()

# Merge with training signals
train_df = signals.merge(vul_clean, on="cve_id", how="left")

# Merge with 2026 test scoreboard
test_df = scoreBoard.merge(vul_clean, on="cve_id", how="left")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "ibm-granite/granite-4.0-micro" 
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype="auto").to(device)
model.eval()

def ask(prompt, max_new_tokens=150):
    chat = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [ ]:
VulDetails = vul[vul['cvss3_score'] > 9.0].sort_values(by='cvss3_score', ascending=False).head(10)
VulDetails = VulDetails[['cve_id', 'cvss3_score', 'description']].copy()

def summarize_cve(desc):
    prompt = (
        "Summarize this vulnerability description in one plain-English sentence. "
        "State what is affected, what an attacker can do, and any key condition. "
        "Do not add information that is not in the text.\n\n"
        f"Description: {desc}"
    )
    return ask(prompt, max_new_tokens=80)

VulDetails['summary'] = VulDetails['description'].apply(summarize_cve)
print(VulDetails[['cve_id', 'cvss3_score', 'summary']].to_string(index=False))